In [1]:
import os
import re
import pandas as pd

## CONFIG

In [2]:
HF_DATASET = "SimulaMet-HOST/Kvasir-VQA"   # will be changed for other datasets
OUT_ROOT = os.path.join(".", "out")

IMAGES_DIR = os.path.join(OUT_ROOT, "images")
META_DIR = os.path.join(OUT_ROOT, "metadata")
MANIFEST_DIR = os.path.join(OUT_ROOT, "manifests")

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

RAW_META_CSV = os.path.join(META_DIR, "metadata_raw.csv")
ENRICHED_META_CSV = os.path.join(META_DIR, "metadata_enriched.csv")
COLONOSCOPY_META_CSV = os.path.join(META_DIR, "colonoscopy_metadata.csv")
IMAGE_MANIFEST_CSV = os.path.join(MANIFEST_DIR, "image_manifest.csv")

COLONOSCOPY_SOURCES = {"Ulcerative Colitis", "Polyps", "Instrument", "Normal"}


## Load Dataset

In [3]:
try:
    from datasets import load_dataset
except Exception:
    raise RuntimeError("Install first: pip install datasets")

ds = load_dataset(HF_DATASET)
raw = ds["raw"]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

## Export Raw Metadata

In [4]:
df = raw.select_columns(["source", "question", "answer", "img_id"]).to_pandas()
df.to_csv(RAW_META_CSV, index=False)
print("Saved:", RAW_META_CSV, "shape:", df.shape)

Saved: ./out/metadata/metadata_raw.csv shape: (58849, 4)


### EXPORT IMAGES (1 per unique img_id) + MANIFEST

In [5]:
unique_img_ids = df["img_id"].dropna().unique().tolist()
print("Unique images:", len(unique_img_ids))

first_idx_by_img = df.reset_index().groupby("img_id")["index"].min().to_dict()

manifest_rows = []
saved = 0
for img_id in unique_img_ids:
    idx = int(first_idx_by_img[img_id])
    out_path = os.path.join(IMAGES_DIR, f"{img_id}.jpg")
    if not os.path.exists(out_path):
        raw[idx]["image"].save(out_path)
        saved += 1
    manifest_rows.append({"img_id": img_id, "image_path": out_path, "exists": os.path.exists(out_path)})

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(IMAGE_MANIFEST_CSV, index=False)
print("Saved:", IMAGE_MANIFEST_CSV, "new_images_saved:", saved)

Unique images: 6500
Saved: ./out/manifests/image_manifest.csv new_images_saved: 0


### ENRICH METADATA 
(question_norm + question_type + answer_type)

In [6]:
def norm_text(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def infer_question_type(q: str) -> str:
    qn = norm_text(q)
    if qn.startswith(("is ", "are ", "was ", "were ", "do ", "does ", "can ", "could ", "has ", "have ")):
        return "Yes/No"
    if qn.startswith(("how many", "number of")):
        return "Counting"
    if qn.startswith(("where", "in which", "at what")):
        return "Location"
    if "color" in qn or "colour" in qn:
        return "Color"
    if qn.startswith(("what", "which")):
        return "Entity"
    return "Other"

def infer_answer_type(a: str) -> str:
    an = norm_text(a)
    if an in {"yes", "no"}:
        return "Yes/No"
    if an in {"none", "na", "n a", "n/a", ""}:
        return "None/NA"
    if re.fullmatch(r"\d+", an):
        return "Numeric"
    if ";" in str(a) or "," in str(a):
        return "List"
    return "Token"

df["question_norm"] = df["question"].astype(str).apply(norm_text)
df["question_type"] = df["question"].astype(str).apply(infer_question_type)
df["answer_type"] = df["answer"].astype(str).apply(infer_answer_type)

df.to_csv(ENRICHED_META_CSV, index=False)
print("Saved:", ENRICHED_META_CSV, "shape:", df.shape)

Saved: ./out/metadata/metadata_enriched.csv shape: (58849, 7)


## Save Colonoscopy Slice

In [7]:
colono = df[df["source"].isin(COLONOSCOPY_SOURCES)].copy()
colono.to_csv(COLONOSCOPY_META_CSV, index=False)
print("Saved:", COLONOSCOPY_META_CSV, "shape:", colono.shape)
print("Sources:", colono["source"].value_counts().to_dict())

Saved: ./out/metadata/colonoscopy_metadata.csv shape: (42126, 7)
Sources: {'Ulcerative Colitis': 16890, 'Polyps': 13539, 'Instrument': 9197, 'Normal': 2500}
